# Analisi esplorativa: uso e percezione dell'intelligenza artificiale

Progetto personale per imparare l'analisi dati in Python.

Il dataset contiene le risposte di **205 persone** a un sondaggio su uso, fiducia e percezione dell'AI. L'obiettivo è capire **chi usa di più l'AI nella vita quotidiana** e se l'uso cambia in base a età, genere, titolo di studio, situazione lavorativa e tempo trascorso sui dispositivi.

L'analisi è solo descrittiva (conteggi, percentuali e medie), quindi i risultati valgono per questo campione e non per l'intera popolazione.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('impact.csv', encoding='cp1254', index_col = 0)

In [ ]:
# Replacing Turkish names
df = df.replace({
    "Ön Bachelor's degree": "Associate degree",
    "Emekliyim": "Retired",
    "Oyun geliştirme": "Game development",
    "Sigortacı": "Insurance agent",
    "Uzman": "Specialist",
})

In [ ]:
# Rename columns
df = df.rename(columns={
    "What is your age range?": "age",
    "What is your gender?": "gender",
    "What is your education level?": "education level",
    "What is your employment status?": "employment status",
    "How often do you use technological devices?": "device hours",
    "Please rate how actively you use AI-powered products in your daily life on a scale from 1 to 5.": "daily AI-powered products usage",
    "Would you like to use more AI products in the future?": "future AI use",
})

In [ ]:
# Missing Values
print(df.isna().sum())
print((df.isna().sum().sum() / df.shape[0]) * 100)

usage = "daily AI-powered products usage"

age_order = ["Under 18 years old", "18-24", "25-34", "35-44", "45 years and older"]
device_order = ["I never use it", "Between 0 to 2 hours per day", "Between 2 to 5 hours per day",
                "Between 5 to 10 hours per day", "More than 10 hours per day"]

Gli unici valori mancanti sono nella colonna dell'occupazione (60 risposte vuote, circa il 29% dei rispondenti), perché la domanda era facoltativa. Non uso quella colonna nell'analisi, quindi non serve gestirli.

## 2. Quanto si usa l'AI in generale?

Ogni rispondente ha valutato da 1 (poco) a 5 (molto) quanto usa prodotti basati sull'AI nella vita quotidiana.

In [ ]:
# General AI Usage
sns.countplot(data=df, x=usage, color="steelblue")
plt.title("Distribuzione dell'uso dell'AI (1 = poco, 5 = molto)")
plt.show()

# Percentage of AI-Usage
print(df[usage].value_counts(normalize=True).sort_index() * 100)

**Osservazione:** l'uso dell'AI è generalmente basso. Il **51%** dei rispondenti dà un voto di 1 o 2, mentre solo il **19%** dà un voto di 4 o 5. Le risposte più frequenti sono 1 e 3 (circa il 30% ciascuna).

## 3. Uso medio dell'AI per gruppo

Per ogni variabile calcolo l'uso medio (da 1 a 5) e il numero di rispondenti in ciascun gruppo. Il numero di rispondenti è importante: le medie dei gruppi piccoli possono cambiare molto anche per poche risposte.

In [ ]:
# Average AI Usage per group
for col, order in [("age", age_order), ("gender", None), ("education level", None),
                   ("employment status", None), ("device hours", device_order)]:
    print(df.groupby(col)[usage].agg(["mean", "count"]).round(2))
    plt.figure(figsize=(9, 5))
    sns.barplot(data=df, x=col, y=usage, order=order, errorbar=None)
    plt.title(f"Uso medio dell'AI per {col}")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

### 3.1 Età

La media da sola non mostra quante persone usano *molto* l'AI. Per questo calcolo, per ogni fascia d'età, la percentuale di rispondenti che ha dato un voto alto (4 o 5).

In [ ]:
# Percentage of high AI usage (4-5) per age group
tab = pd.crosstab(df["age"], df[usage], normalize="index").reindex(age_order) * 100
tab[[4, 5]].sum(axis=1)

**Osservazione:** l'età è la variabile con l'effetto più evidente. Nella fascia **18–24 anni il 32%** dà un voto alto (4–5), contro **meno del 10%** sopra i 35 anni. I più giovani hanno anche l'uso medio più alto (3,0 su 5), mentre dai 25 anni in su la media scende a circa 2.

### 3.2 Situazione lavorativa

In [ ]:
# Average AI usage per employment status
df.groupby("employment status")[usage].mean()

**Osservazione:** gli **studenti** hanno l'uso medio più alto (**2,9** su 5), mentre i **lavoratori** si fermano a **2,0**. Il gruppo dei pensionati ha la media più bassa (1,5), ma conta solo 6 persone, quindi il dato è poco affidabile.

### 3.3 Ore al giorno sui dispositivi

In [ ]:
# Average AI usage per device hours
df.groupby("device hours")[usage].mean().reindex(device_order)

**Osservazione:** l'uso dell'AI cresce in modo regolare con il tempo trascorso sui dispositivi: la media sale da **1,8** (0–2 ore al giorno) a **2,9** (oltre 10 ore). Il gruppo "I never use it" ha media 1,0 ma conta solo 3 persone.

### 3.4 Genere e titolo di studio

In [ ]:
# Average AI usage per gender and education level
print(df.groupby("gender")[usage].mean().round(2))
print(df.groupby("education level")[usage].mean().round(2))

**Osservazione:** genere e titolo di studio non sembrano fare differenza. Donne (2,46) e uomini (2,38) hanno medie quasi identiche, e anche tra i titoli di studio le medie sono vicine. Il valore più basso (laurea breve, 1,87) riguarda solo 15 persone.

## 4. Conclusioni

- L'uso dell'AI nel campione è **generalmente basso**: metà dei rispondenti dà un voto di 1 o 2 su 5.
- Le differenze più evidenti riguardano **età** e **situazione lavorativa**: i giovani tra i 18 e i 24 anni e gli studenti usano l'AI più degli altri.
- Anche il **tempo sui dispositivi** va di pari passo con l'uso dell'AI.
- **Genere** e **titolo di studio** non mostrano differenze rilevanti.

